In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
batch_size = 64
learning_rate = 0.001
epochs = 5

epsilons = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
model_path = "/kaggle/working/mnist_cnn.pth"

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="/kaggle/working/data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
examples = enumerate(train_loader)
batch_idx, (example_data, example_targets) = next(examples)

fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i in range(6):
    axes[i].imshow(example_data[i][0], cmap="gray")
    axes[i].set_title(f"Label: {example_targets[i].item()}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(model)

In [ ]:
def train_model(model, train_loader, optimizer, criterion, device, epochs):
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}%")

In [ ]:
def evaluate_clean(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100 * correct / total
    return accuracy

In [ ]:
train_model(model, train_loader, optimizer, criterion, device, epochs)

In [ ]:
clean_acc = evaluate_clean(model, DataLoader(test_dataset, batch_size=64, shuffle=False), device)
print(f"Clean test accuracy: {clean_acc:.2f}%")

torch.save(model.state_dict(), model_path)
print("Model saved to:", model_path)

In [ ]:
def fgsm_attack(image, epsilon, data_grad):
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image

In [ ]:
def test_fgsm(model, test_loader, device, epsilon):
    model.eval()
    correct = 0
    total = 0
    examples = []

    for image, label in test_loader:
        image, label = image.to(device), label.to(device)
        image.requires_grad = True

        output = model(image)
        init_pred = output.argmax(dim=1)

        # Only attack if model got it right originally
        if init_pred.item() != label.item():
            continue

        loss = criterion(output, label)
        model.zero_grad()
        loss.backward()

        data_grad = image.grad.data
        perturbed_image = fgsm_attack(image, epsilon, data_grad)

        output_adv = model(perturbed_image)
        final_pred = output_adv.argmax(dim=1)

        correct += (final_pred == label).sum().item()
        total += 1

        if len(examples) < 5:
            examples.append((
                label.item(),
                init_pred.item(),
                final_pred.item(),
                image.detach().cpu().squeeze(),
                perturbed_image.detach().cpu().squeeze()
            ))

    accuracy = correct / total if total > 0 else 0
    return accuracy, examples

In [ ]:
accuracies = []
all_examples = []

for epsilon in epsilons:
    acc, ex = test_fgsm(model, test_loader, device, epsilon)
    accuracies.append(acc)
    all_examples.append(ex)
    print(f"Epsilon: {epsilon:.2f} \t Accuracy: {acc:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epsilons, accuracies, marker="o")
plt.title("FGSM Attack: Accuracy vs Epsilon")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

In [ ]:
fig, axes = plt.subplots(len(epsilons), 5, figsize=(12, 16))
fig.suptitle("Adversarial Examples at Different Epsilon Values", fontsize=16)

for row_idx, epsilon in enumerate(epsilons):
    examples = all_examples[row_idx]
    
    for col_idx in range(5):
        ax = axes[row_idx, col_idx]
        
        if col_idx < len(examples):
            true_label, init_pred, final_pred, orig, adv = examples[col_idx]
            ax.imshow(adv, cmap="gray")
            ax.set_title(f"e={epsilon:.2f}\n{init_pred}->{final_pred}")
            ax.axis("off")
        else:
            ax.axis("off")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
chosen_index = 3  # corresponds to epsilon = 0.15
chosen_epsilon = epsilons[chosen_index]
examples = all_examples[chosen_index]

fig, axes = plt.subplots(len(examples), 2, figsize=(6, 12))
fig.suptitle(f"Original vs Adversarial Images (epsilon = {chosen_epsilon})", fontsize=16)

for i, (true_label, init_pred, final_pred, orig, adv) in enumerate(examples):
    axes[i, 0].imshow(orig, cmap="gray")
    axes[i, 0].set_title(f"Original\nTrue: {true_label}, Pred: {init_pred}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(adv, cmap="gray")
    axes[i, 1].set_title(f"Adversarial\nPred: {final_pred}")
    axes[i, 1].axis("off")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
results_path = "/kaggle/working/fgsm_results.txt"

with open(results_path, "w") as f:
    f.write("FGSM Accuracy Results\n")
    f.write("=====================\n")
    f.write(f"Clean test accuracy: {clean_acc:.2f}%\n\n")
    for epsilon, acc in zip(epsilons, accuracies):
        f.write(f"Epsilon: {epsilon:.2f}, Accuracy: {acc:.4f}\n")

print("Results saved to:", results_path)

In [ ]:
import copy

adv_model = copy.deepcopy(model).to(device)
adv_optimizer = optim.Adam(adv_model.parameters(), lr=learning_rate)
adv_train_epsilon = 0.15
adv_epochs = 3


In [ ]:
def adversarial_train(model, train_loader, optimizer, criterion, device, epochs, epsilon):
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        clean_correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            images.requires_grad = True

            # Step 1: compute gradients on clean images
            optimizer.zero_grad()
            outputs = model(images)
            clean_loss = criterion(outputs, labels)
            clean_loss.backward()

            data_grad = images.grad.detach()
            adv_images = fgsm_attack(images.detach(), epsilon, data_grad)

            # Step 2: update model using adversarial images
            optimizer.zero_grad()
            adv_outputs = model(adv_images)
            adv_loss = criterion(adv_outputs, labels)
            adv_loss.backward()
            optimizer.step()

            running_loss += adv_loss.item()
            clean_correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / len(train_loader)
        epoch_clean_acc = 100 * clean_correct / total
        history.append((epoch_loss, epoch_clean_acc))
        print(f"Adv Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Clean Train Acc: {epoch_clean_acc:.2f}%")

    return history


In [ ]:
adv_history = adversarial_train(
    adv_model,
    train_loader,
    adv_optimizer,
    criterion,
    device,
    adv_epochs,
    adv_train_epsilon,
)


In [ ]:
adv_clean_acc = evaluate_clean(adv_model, DataLoader(test_dataset, batch_size=64, shuffle=False), device)
print(f"Adversarially trained model clean test accuracy: {adv_clean_acc:.2f}%")


In [ ]:
adv_accuracies = []
adv_all_examples = []

for epsilon in epsilons:
    acc, ex = test_fgsm(adv_model, test_loader, device, epsilon)
    adv_accuracies.append(acc)
    adv_all_examples.append(ex)
    print(f"Adversarially trained model - Epsilon: {epsilon:.2f} 	 Accuracy: {acc:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epsilons, accuracies, marker="o", label="Baseline model")
plt.plot(epsilons, adv_accuracies, marker="s", label="Adversarially trained model")
plt.title("FGSM Robustness Comparison")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
baseline_clean_fraction = clean_acc / 100.0
adv_clean_fraction = adv_clean_acc / 100.0

print(f"Baseline clean accuracy (fraction): {baseline_clean_fraction:.4f}")
print(f"Adv-trained clean accuracy (fraction): {adv_clean_fraction:.4f}")

print("
FGSM accuracy comparison")
for epsilon, base_acc, adv_acc in zip(epsilons, accuracies, adv_accuracies):
    print(f"Epsilon {epsilon:.2f}: baseline={base_acc:.4f}, adv_trained={adv_acc:.4f}")


In [ ]:
chosen_index = 3  # epsilon = 0.15
chosen_epsilon = epsilons[chosen_index]

baseline_examples = all_examples[chosen_index]
adv_examples = adv_all_examples[chosen_index]

num_rows = min(5, max(len(baseline_examples), len(adv_examples)))
fig, axes = plt.subplots(num_rows, 4, figsize=(10, 2.5 * num_rows))
fig.suptitle(f"Baseline vs Adversarially Trained Model (epsilon = {chosen_epsilon})", fontsize=16)

if num_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for i in range(num_rows):
    # Baseline examples
    if i < len(baseline_examples):
        true_label, init_pred, final_pred, orig, adv = baseline_examples[i]
        axes[i, 0].imshow(orig, cmap="gray")
        axes[i, 0].set_title(f"Baseline Orig\nT:{true_label} P:{init_pred}")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(adv, cmap="gray")
        axes[i, 1].set_title(f"Baseline Adv\nP:{final_pred}")
        axes[i, 1].axis("off")
    else:
        axes[i, 0].axis("off")
        axes[i, 1].axis("off")

    # Adversarially trained examples
    if i < len(adv_examples):
        true_label, init_pred, final_pred, orig, adv = adv_examples[i]
        axes[i, 2].imshow(orig, cmap="gray")
        axes[i, 2].set_title(f"AdvTrain Orig\nT:{true_label} P:{init_pred}")
        axes[i, 2].axis("off")
        axes[i, 3].imshow(adv, cmap="gray")
        axes[i, 3].set_title(f"AdvTrain Adv\nP:{final_pred}")
        axes[i, 3].axis("off")
    else:
        axes[i, 2].axis("off")
        axes[i, 3].axis("off")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


In [ ]:
final_results_path = "/kaggle/working/final_project_results.txt"

with open(final_results_path, "w") as f:
    f.write("Adversarial Machine Learning Project Results\n")
    f.write("========================================\n")
    f.write(f"Baseline clean accuracy: {clean_acc:.2f}%\n")
    f.write(f"Adversarially trained clean accuracy: {adv_clean_acc:.2f}%\n\n")
    f.write("FGSM accuracy comparison\n")
    f.write("epsilon	baseline	adv_trained\n")
    for epsilon, base_acc, adv_acc in zip(epsilons, accuracies, adv_accuracies):
        f.write(f"{epsilon:.2f}	{base_acc:.4f}	{adv_acc:.4f}\n")

print("Final project results saved to:", final_results_path)
